# Phase 2 — Train MuRIL Banking Intent Classifier
Trains `google/muril-base-cased` on the 15 banking intents created by `make_banking_dataset.py`.
Uses romanization + chat-style data augmentation (`common.py`).
Outputs `muril_banking/` and `muril_banking.zip` for deployment in `model_service/`.

In [ ]:
# ---- CONFIG ----
MODEL, LR, EPOCHS, BS, MAX_LEN, SEED = 'google/muril-base-cased', 2e-5, 5, 32, 64, 42
TRAIN_FILE = 'banking_train.csv'
LABELS_FILE = 'banking_labels.json'

In [ ]:
import os, glob, json, random, time, shutil, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from common import augment, is_dev

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
OUT = '/kaggle/working' if ON_KAGGLE else './muril_banking_out'
os.makedirs(OUT, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
train = pd.read_csv(TRAIN_FILE, keep_default_na=False)
classes = np.sort(train.target.unique()); c2i = {c: i for i, c in enumerate(classes)}
train['y'] = train.target.map(c2i)
train['roman'] = ~train.query_text.map(is_dev)
NC = len(classes)
print(f'Training rows: {len(train)} | Classes: {NC}')

In [ ]:
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(MODEL)
labels_data = json.load(open(LABELS_FILE))
id2label = {i: n for i, n in enumerate(labels_data['names'])}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=NC, id2label=id2label, label2id=label2id
).to(DEV)

def collate_fn(b):
    enc = tok([x[0] for x in b], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    enc['labels'] = torch.tensor([x[1] for x in b])
    return enc

dl = torch.utils.data.DataLoader(list(zip(train.query_text, train.y)), batch_size=BS, shuffle=True, collate_fn=collate_fn)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = len(dl) * EPOCHS
sch = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler('cuda') if DEV == 'cuda' else None
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
print('Starting fine-tuning...')
t0 = time.time()
for ep in range(EPOCHS):
    model.train(); run = 0
    for i, b in enumerate(dl):
        b = {k: v.to(DEV) for k, v in b.items()}
        y = b.pop('labels')
        if scaler:
            with torch.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(**b).logits.float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        else:
            loss = lossf(model(**b).logits, y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        run += loss.item()
    print(f'epoch {ep+1}/{EPOCHS} loss {run / len(dl):.4f} ({time.time() - t0:.0f}s)', flush=True)

In [ ]:
# Export half-precision model and labels for CPU inference service
save_dir = f'{OUT}/muril_banking'
os.makedirs(save_dir, exist_ok=True)
model.half().save_pretrained(save_dir) if DEV == 'cuda' else model.save_pretrained(save_dir)
tok.save_pretrained(save_dir)
shutil.copy(LABELS_FILE, f'{save_dir}/labels.json')
shutil.make_archive(f'{OUT}/muril_banking', 'zip', save_dir)
print('Model saved successfully:', save_dir, '| Archive:', f'{OUT}/muril_banking.zip')